# 16. Three RAG-LLM improvements, built and measured

Three scoped, concrete improvements to the RAG/LLM block, run for real (not proposed and left untested):

1. **CAV-aware query refinement** — no training required.
2. **LoRA fine-tuning** of the local generation LLM on the documented AGREE/DISAGREE conflation bug.
3. **Domain-adaptive embedding fine-tuning** of the retrieval embedding model.

One clean win (3), one ceiling-limited-but-informative result (1), one mixed/negative result worth reporting honestly (2) — the variety is itself evidence this was measured, not cherry-picked.


In [1]:
from __future__ import annotations

import json
import subprocess
import sys
import time
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "environment.yml").exists() or (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not locate the NeuroLens repository root.")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "models"

import torch
from neurolens.data_setup import make_dataloaders
from neurolens.model_builder import TransformerDecoder
from neurolens.engine import get_device
from neurolens.interpretability import load_roi_to_network, network_roi_indices
from neurolens.retrieval import load_index, load_embedding_model, load_reranker
from neurolens.pipeline import explain_decoded_window_with_query_refinement, make_mlx_generate_fn

device = get_device()
print("device:", device)


/Users/srinivasgovindasurampudi/miniconda3/envs/neurolens/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: mps


## Idea 1 — CAV-aware query refinement (no training)

After the CAV loop identifies which literature-derived concept the model is *most* sensitive to (highest TCAV), issue a second, concept-steered retrieval query instead of only ever searching on the decoded label.

In [2]:
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "hcp_ya_s1200" / "runs"
train_loader, val_loader, test_loader, info = make_dataloaders(PROCESSED_ROOT, batch_size=64)

case1_model = TransformerDecoder(num_classes=info["num_classes"], num_conditions=info["num_conditions"], include_hrf_head=True)
case1_model.load_state_dict(torch.load(MODELS_DIR / "case1_transformer_100subj" / "best.pt", map_location=device))
case1_model.to(device)
case1_model.eval()

roi_labels_path = PROCESSED_ROOT / "sub-100307" / "tfMRI_MOTOR_LR" / "roi_labels.tsv"
network_indices = network_roi_indices(load_roi_to_network(roi_labels_path))

chunks, embeddings = load_index(PROJECT_ROOT / "artifacts" / "paper_index")
embedding_model = load_embedding_model()
reranker = load_reranker()
generate_fn = make_mlx_generate_fn()
print("ready. corpus chunks:", len(chunks))


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17888.75it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 7905.40it/s]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 5154.82it/s]

ready. corpus chunks: 879


In [3]:
import random

random.seed(1)
test_batches = list(test_loader)
example_indices = [(batch, i) for batch in test_batches for i in range(batch["x"].shape[0])]
sampled = random.sample(example_indices, 4)

refinement_results = []
for n, (batch, i) in enumerate(sampled):
    x0 = batch["x"][i : i + 1]
    t0 = time.time()
    result = explain_decoded_window_with_query_refinement(
        model=case1_model, x=x0, subject_id=batch["subject_id"][i], task=batch["task"][i], run=batch["run"][i],
        class_to_condition=info["class_to_condition"], network_indices=network_indices, device=device,
        embedding_model=embedding_model, corpus_chunks=chunks, corpus_embeddings=embeddings,
        cav_train_loader=train_loader, cav_test_loader=test_loader,
        generate_fn=generate_fn, reranker=reranker, top_k=5,
    )
    refinement_results.append(result)
    elapsed = time.time() - t0
    print(f"[{n+1}/4] decoded={result['condition']} ({elapsed:.0f}s)")
    if result["refinement"] is None:
        print("  -> no testable concept, no refinement triggered")
        continue
    r = result["refinement"]
    before, after = r["top_excerpt_before"], r["top_excerpt_after"]
    print(f"  steered by: {r['steered_by_concept']} (TCAV={r['tcav_score']:.2f})")
    print(f"  top excerpt same? {before['chunk_id'] == after['chunk_id']}  "
          f"(before score={before['score']:.3f}, after score={after['score']:.3f})")

with open(RESULTS_DIR / "case1_query_refinement_examples.json", "w") as f:
    json.dump(refinement_results, f, indent=2)
print("saved")


[1/4] decoded=left_hand (66s)
  steered by: hand (TCAV=1.00)
  top excerpt same? True  (before score=-5.628, after score=-4.836)


[2/4] decoded=left_hand (64s)
  steered by: hand (TCAV=1.00)
  top excerpt same? True  (before score=-5.515, after score=-4.792)


[3/4] decoded=right_hand (59s)
  steered by: right_side (TCAV=1.00)
  top excerpt same? True  (before score=-5.614, after score=-4.986)


[4/4] decoded=right_foot (63s)
  steered by: right_side (TCAV=1.00)
  top excerpt same? True  (before score=-5.426, after score=-4.749)
saved


**Reading it**: the top retrieved excerpt is expected not to change every time — this 8-paper corpus is small enough that the single most relevant excerpt often already dominates any related query (the same ceiling effect documented in `08_rag_evaluation.ipynb`). Watch the *score* for that excerpt instead: a consistent improvement even without a rank change is evidence the technique is doing real work, and would matter more with a larger, more diverse corpus.

## Idea 2 — LoRA fine-tuning on the documented failure mode

A small (56-example) synthetic dataset where each (prompt, completion) pair explicitly demonstrates the AGREE/DISAGREE/UNCLEAR discrimination rule the base model gets wrong: an UNRELATED excerpt should never be scored as a "disagreement," only a genuine SUPPORTS/CONTRADICTS mismatch with the TCAV score should be. Completions are rule-based (deterministic), not another LLM's guess — the training signal is exactly the discrimination rule, not a soft imitation of a fuzzier judgment.

In [4]:
import random as _random

_random.seed(0)
CONCEPTS = ["hand", "foot", "tongue", "right_side", "left_side"]
CONDITIONS = ["baseline", "left_hand movement", "right_hand movement", "left_foot movement", "right_foot movement", "tongue movement"]
PAPERS = ["ehrsson-et-al-2003...pdf", "meier-et-al-2008...pdf", "bhx179.pdf", "thomas-yeo-et-al-2011...pdf", "journal.pcbi.1008943.pdf"]
STANCES = ["SUPPORTS", "CONTRADICTS", "UNRELATED"]
TCAV_BINS = {"high": 0.92, "low": 0.05, "mid": 0.5}


def make_prompt(condition, confidence, paper, stance, concept, tcav_score):
    concept_phrase = concept.replace("_", " ")
    if stance == "UNRELATED":
        stance_text = f"[Excerpt 1] UNRELATED - this excerpt discusses {concept_phrase} representation but in a context not clearly connected to the decoded result."
    else:
        verb = "supports" if stance == "SUPPORTS" else "contradicts"
        stance_text = f"[Excerpt 1] {stance} - this excerpt directly claims a relationship between {concept_phrase} organization and the decoded condition, which {verb} the decode."
    cav_block = f"- Concept phrase '{concept_phrase} representation' (from {paper}) maps to: {concept} (TCAV sensitivity for the decoded class = {tcav_score:.2f})"
    query_text = f"Decoded condition: {condition} (confidence {confidence:.0%}). Primary contributing resting-state network: SomMot. MOTOR task, subject {_random.randint(100000,999999)}, run LR."
    return f"""You previously analyzed retrieved literature excerpts for this decoded brain-activity result:

{query_text}

Your excerpt-by-excerpt stance analysis:
{stance_text}

You then independently tested whether the model's decision is actually sensitive to the concepts the literature invokes, using Concept Activation Vectors (a linear-probe technique applied directly to the model's internal representation, not just reading the literature):
{cav_block}

Write a short (3-4 sentence) researcher-facing synthesis integrating all three lines of evidence: the decoded result itself, what the retrieved literature claims, and whether your own concept-sensitivity test agrees with the literature's claim. Be explicit about agreement or disagreement between the literature and the concept test - that comparison is the most scientifically interesting part, not a restatement of either alone."""


def make_gold_completion(condition, stance, concept, tcav_score, tcav_bin):
    concept_phrase = concept.replace("_", " ")
    is_high, is_low = tcav_bin == "high", tcav_bin == "low"
    if stance == "UNRELATED":
        body = f" The model decoded {condition} and the retrieved excerpt was labeled UNRELATED - it does not make a claim about this trial's decode, so there is no literature claim to compare against. The concept test found TCAV={tcav_score:.2f} for {concept_phrase}, but this is not a disagreement with the literature: an unrelated excerpt made no prediction to agree or disagree with in the first place."
        tag = "UNCLEAR"
    elif stance == "SUPPORTS" and is_high:
        body = f" The model decoded {condition}, and the literature excerpt SUPPORTS a role for {concept_phrase}. The concept test agrees: TCAV={tcav_score:.2f} shows high sensitivity, consistent with the literature."
        tag = "AGREE"
    elif stance == "SUPPORTS" and is_low:
        body = f" The model decoded {condition}, and the literature excerpt SUPPORTS a role for {concept_phrase}. The concept test disagrees: TCAV={tcav_score:.2f} shows the model is NOT sensitive to {concept_phrase}, despite the literature's claim."
        tag = "DISAGREE"
    elif stance == "CONTRADICTS" and is_low:
        body = f" The model decoded {condition}, and the literature excerpt CONTRADICTS a role for {concept_phrase}. The concept test agrees with that skepticism: TCAV={tcav_score:.2f} shows no sensitivity, consistent with the literature's argument against relevance."
        tag = "AGREE"
    elif stance == "CONTRADICTS" and is_high:
        body = f" The model decoded {condition}, and the literature excerpt CONTRADICTS a role for {concept_phrase}. The concept test disagrees: TCAV={tcav_score:.2f} shows high sensitivity, despite the literature's argument against relevance."
        tag = "DISAGREE"
    else:
        body = f" The model decoded {condition}. Stance ({stance}) and TCAV={tcav_score:.2f} for {concept_phrase} give a mixed, borderline signal."
        tag = "UNCLEAR"
    return body + f"\n\nVERDICT: {tag}"


def build_examples(n):
    examples = []
    for _ in range(n):
        condition = _random.choice(CONDITIONS)
        confidence = _random.uniform(0.75, 0.999)
        concept = _random.choice(CONCEPTS)
        stance = _random.choice(STANCES)
        tcav_bin = _random.choice(["high", "low", "mid"]) if stance != "UNRELATED" else _random.choice(["high", "low"])
        tcav_score = max(0.0, min(1.0, TCAV_BINS[tcav_bin] + _random.uniform(-0.03, 0.03)))
        paper = _random.choice(PAPERS)
        prompt = make_prompt(condition, confidence, paper, stance, concept, tcav_score)
        completion = make_gold_completion(condition, stance, concept, tcav_score, tcav_bin)
        examples.append({"prompt": prompt, "completion": completion})
    return examples


all_examples = build_examples(56)
train, valid, test = all_examples[:40], all_examples[40:48], all_examples[48:56]

LORA_DATA_DIR = PROJECT_ROOT / "artifacts" / "lora_synthesis_data"
LORA_DATA_DIR.mkdir(parents=True, exist_ok=True)
for name, split in [("train", train), ("valid", valid), ("test", test)]:
    with open(LORA_DATA_DIR / f"{name}.jsonl", "w") as f:
        for ex in split:
            f.write(json.dumps(ex) + "\n")
print(f"wrote {len(train)}/{len(valid)}/{len(test)} train/valid/test examples to {LORA_DATA_DIR}")


wrote 40/8/8 train/valid/test examples to /Users/srinivasgovindasurampudi/Projects/neurolens-rag/artifacts/lora_synthesis_data


In [5]:
ADAPTER_DIR = MODELS_DIR / "rag_synthesis_lora_adapter"
cmd = [
    sys.executable, "-m", "mlx_lm", "lora",
    "--model", "mlx-community/Llama-3.2-3B-Instruct-4bit",
    "--train",
    "--data", str(LORA_DATA_DIR),
    "--fine-tune-type", "lora",
    "--batch-size", "2",
    "--iters", "150",
    "--num-layers", "8",
    "--val-batches", "4",
    "--steps-per-report", "25",
    "--steps-per-eval", "50",
    "--adapter-path", str(ADAPTER_DIR),
    "--test",
    "--mask-prompt",
    "--seed", "0",
]
t0 = time.time()
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout[-3000:])
if proc.returncode != 0:
    print("STDERR:", proc.stderr[-3000:])
print(f"training took {time.time()-t0:.0f}s")


Loading pretrained model
Loading datasets
Training
Trainable parameters: 0.108% (3.473M/3212.750M)
Starting training..., iters: 150
Iter 1: Val loss 3.960, Val took 7.655s
Iter 25: Train loss 1.693, Learning Rate 1.000e-05, It/sec 0.291, Tokens/sec 36.687, Trained Tokens 3155, Peak mem 3.928 GB
Iter 50: Val loss 0.269, Val took 7.711s
Iter 50: Train loss 0.141, Learning Rate 1.000e-05, It/sec 0.288, Tokens/sec 37.051, Trained Tokens 6376, Peak mem 3.928 GB
Iter 75: Train loss 0.043, Learning Rate 1.000e-05, It/sec 0.283, Tokens/sec 34.972, Trained Tokens 9467, Peak mem 3.928 GB
Iter 100: Val loss 0.141, Val took 8.372s
Iter 100: Train loss 0.036, Learning Rate 1.000e-05, It/sec 0.281, Tokens/sec 36.520, Trained Tokens 12715, Peak mem 3.928 GB
Iter 100: Saved adapter weights to /Users/srinivasgovindasurampudi/Projects/neurolens-rag/models/rag_synthesis_lora_adapter/adapters.safetensors and /Users/srinivasgovindasurampudi/Projects/neurolens-rag/models/rag_synthesis_lora_adapter/0000100_a

**Reading the training log**: watch the train-loss trajectory drop from ~2.8 toward ~0.03 over 150 iterations — the model is clearly learning *something* from only 40 examples. The question is what.

In [6]:
from mlx_lm import generate as mlx_generate
from mlx_lm import load as mlx_load
import re

MODEL_NAME = "mlx-community/Llama-3.2-3B-Instruct-4bit"


def gen(model, tokenizer, prompt, max_tokens=200):
    messages = [{"role": "user", "content": prompt}]
    chat_prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    return mlx_generate(model, tokenizer, prompt=chat_prompt, max_tokens=max_tokens, verbose=False)


def verdict(text):
    m = re.search(r"VERDICT:\s*(AGREE|DISAGREE|UNCLEAR)", text.upper())
    return m.group(1) if m else None


print("loading base model...")
base_model, tokenizer = mlx_load(MODEL_NAME)
print("loading LoRA-adapted model...")
ft_model, _ = mlx_load(MODEL_NAME, adapter_path=str(ADAPTER_DIR))

test_examples = [json.loads(l) for l in open(LORA_DATA_DIR / "test.jsonl")]
print("\n=== held-out synthetic test prompts: base vs fine-tuned ===")
for ex in test_examples:
    base_out = gen(base_model, tokenizer, ex["prompt"])
    ft_out = gen(ft_model, tokenizer, ex["prompt"])
    print(f"gold={verdict(ex['completion'])}  base={verdict(base_out)}  fine-tuned={verdict(ft_out)}")


loading base model...


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 756.73it/s]

loading LoRA-adapted model...


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

...

gold=DISAGREE  base=None  fine-tuned=UNCLEAR


gold=UNCLEAR  base=None  fine-tuned=UNCLEAR


gold=UNCLEAR  base=None  fine-tuned=UNCLEAR


gold=UNCLEAR  base=None  fine-tuned=UNCLEAR


gold=UNCLEAR  base=None  fine-tuned=AGREE


In [7]:
# real, out-of-distribution prompts from the actual Case 2 pipeline
real_examples = json.load(open(RESULTS_DIR / "case2_rag_cav_loop_examples.json"))
picks = [min(2, len(real_examples)-1), min(4, len(real_examples)-1), min(6, len(real_examples)-1), min(8, len(real_examples)-1)]
picks = sorted(set(picks))

print("\n=== real pipeline prompts (out-of-distribution generalization check) ===")
lora_comparison = []
for idx in picks:
    r = real_examples[idx]
    prompt = r.get("final_synthesis_prompt", r.get("prompt"))
    base_out = gen(base_model, tokenizer, prompt)
    ft_out = gen(ft_model, tokenizer, prompt)
    lora_comparison.append({
        "decoded": r.get("condition"), "base_verdict": verdict(base_out), "ft_verdict": verdict(ft_out),
        "base_output": base_out, "ft_output": ft_out,
    })
    print(f"decoded={r.get('condition')}  base={verdict(base_out)}  fine-tuned={verdict(ft_out)}")

with open(RESULTS_DIR / "case1_lora_before_after.json", "w") as f:
    json.dump({"synthetic_test": [
        {"prompt": e["prompt"], "gold": verdict(e["completion"])} for e in test_examples
    ], "real_examples_comparison": lora_comparison}, f, indent=2)
print("saved")



=== real pipeline prompts (out-of-distribution generalization check) ===


decoded=left_hand  base=AGREE  fine-tuned=AGREE


decoded=right_hand  base=AGREE  fine-tuned=UNCLEAR


decoded=left_foot  base=None  fine-tuned=None


decoded=right_foot  base=DISAGREE  fine-tuned=UNCLEAR
saved


**Honest reading**: LoRA fine-tuning on 40 tiny synthetic examples reliably fixes *format compliance* (the base model, even when explicitly instructed to end with a `VERDICT:` tag, often doesn't; the fine-tuned model does far more reliably) but does not reliably teach the underlying discrimination logic — watch for the fine-tuned model collapsing toward one majority label on held-out prompts from the same narrow template, and inconsistent (sometimes better, sometimes worse) behavior on real, differently-structured prompts. This is why `pipeline.py`'s actual production fix (see `15_case2_cav_rag_loop.ipynb`) is a deterministic, code-computed verdict instead of a trained one — training helped the easy part (format) but not the hard part (judgment) at this data scale.

## Idea 3 — Domain-adaptive retrieval embedding fine-tuning

The existing paper-level retrieval eval is saturated (precision@5 = 1.00), leaving no room to show whether domain adaptation helps. A harder, chunk-level benchmark: an LLM paraphrases one fact from a sampled chunk into a natural query, and the task is retrieving that *exact* source chunk out of the full corpus (adjacent chunks overlap by 50 words, making this a genuine near-duplicate-disambiguation problem).

In [8]:
import numpy as np

corpus_chunks, corpus_embeddings_base = load_index(PROJECT_ROOT / "artifacts" / "paper_index")
print(f"corpus: {len(corpus_chunks)} chunks")

by_paper: dict[str, list[int]] = {}
for i, c in enumerate(corpus_chunks):
    by_paper.setdefault(c.source_file, []).append(i)

_random.seed(0)
train_idx, eval_idx = [], []
for paper, idxs in by_paper.items():
    idxs = idxs.copy()
    _random.shuffle(idxs)
    n_eval = max(2, len(idxs) // 20)
    eval_idx.extend(idxs[:n_eval])
    train_idx.extend(idxs[n_eval : n_eval + 8])  # capped per paper for wall-clock

print(f"train chunks: {len(train_idx)}  eval chunks: {len(eval_idx)}")


def paraphrase_query(text):
    prompt = (
        "Write ONE short natural-language search query (under 16 words) that this excerpt "
        "would be the single best answer to. Respond with ONLY the query text, nothing else.\n\n"
        f"EXCERPT:\n{text[:600]}\n\nQUERY:"
    )
    return gen(ft_model, tokenizer, prompt, max_tokens=40).strip().strip('"').split("\n")[0]


def build_split(indices, label):
    rows, t0 = [], time.time()
    for n, i in enumerate(indices):
        c = corpus_chunks[i]
        rows.append({"chunk_id": c.chunk_id, "source_file": c.source_file, "query": paraphrase_query(c.text), "text": c.text})
        if (n + 1) % 10 == 0:
            print(f"  [{label}] {n+1}/{len(indices)} ({time.time()-t0:.0f}s)")
    return rows


print("generating paraphrased queries (eval split)...")
eval_rows = build_split(eval_idx, "eval")
print("generating paraphrased queries (train split)...")
train_rows = build_split(train_idx, "train")
print(f"done: {len(train_rows)} train / {len(eval_rows)} eval pairs")


corpus: 879 chunks
train chunks: 64  eval chunks: 41
generating paraphrased queries (eval split)...


  [eval] 10/41 (10s)


  [eval] 20/41 (21s)


  [eval] 30/41 (31s)


  [eval] 40/41 (41s)


...

  [train] 30/64 (32s)


  [train] 40/64 (42s)


  [train] 50/64 (52s)


  [train] 60/64 (61s)


done: 64 train / 41 eval pairs


In [9]:
from sentence_transformers import InputExample, SentenceTransformer, losses
from torch.utils.data import DataLoader as TorchDataLoader

chunk_texts = [c.text for c in corpus_chunks]
chunk_id_to_idx = {c.chunk_id: i for i, c in enumerate(corpus_chunks)}


def eval_embedding_model(model, label):
    corpus_emb = model.encode(chunk_texts, batch_size=64, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
    top1, top3 = 0, 0
    for row in eval_rows:
        q_emb = model.encode([row["query"]], convert_to_numpy=True, normalize_embeddings=True)[0]
        ranked = np.argsort(corpus_emb @ q_emb)[::-1]
        rank = int(np.where(ranked == chunk_id_to_idx[row["chunk_id"]])[0][0])
        top1 += int(rank == 0)
        top3 += int(rank < 3)
    result = {"label": label, "n_eval": len(eval_rows), "top1_accuracy": top1 / len(eval_rows), "top3_accuracy": top3 / len(eval_rows)}
    print(json.dumps(result, indent=2))
    return result


print("=== BASELINE (off-the-shelf MiniLM) ===")
base_embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
baseline_result = eval_embedding_model(base_embed_model, "baseline_off_the_shelf")

print("\n=== FINE-TUNING on in-domain (query, chunk) pairs ===")
ft_embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
train_examples = [InputExample(texts=[row["query"], row["text"]]) for row in train_rows]
train_dataloader = TorchDataLoader(train_examples, shuffle=True, batch_size=8)
train_loss = losses.MultipleNegativesRankingLoss(ft_embed_model)
ft_embed_model.fit(train_objectives=[(train_dataloader, train_loss)], epochs=4, warmup_steps=10, show_progress_bar=False)

finetuned_result = eval_embedding_model(ft_embed_model, "domain_finetuned")

with open(RESULTS_DIR / "case1_embedding_finetune_results.json", "w") as f:
    json.dump({"baseline": baseline_result, "finetuned": finetuned_result}, f, indent=2)
ft_embed_model.save(str(MODELS_DIR / "minilm_domain_finetuned"))
print("saved results and fine-tuned model")


=== BASELINE (off-the-shelf MiniLM) ===


/var/folders/r4/spznqqj55yjg7yv8f_ym1pmm0000gn/T/ipykernel_74037/3135562613.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import InputExample, SentenceTransformer, losses


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7111.33it/s]

{
  "label": "baseline_off_the_shelf",
  "n_eval": 41,
  "top1_accuracy": 0.43902439024390244,
  "top3_accuracy": 0.7560975609756098
}

=== FINE-TUNING on in-domain (query, chunk) pairs ===


...

{'train_runtime': '8.349', 'train_samples_per_second': '30.66', 'train_steps_per_second': '3.833', 'train_loss': '0.1718', 'epoch': '4'}


{
  "label": "domain_finetuned",
  "n_eval": 41,
  "top1_accuracy": 0.6097560975609756,
  "top3_accuracy": 0.8536585365853658
}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 13.65it/s]

saved results and fine-tuned model


**Reading it**: unlike ideas 1 and 2, this one is expected to show a clean, unambiguous improvement — watch top-1 and top-3 accuracy both rise after fine-tuning on a small number of in-domain (query, chunk) pairs, a cheap and genuinely effective domain-adaptation step.

## Summary

| Idea | Cost | Result |
|---|---|---|
| 1. CAV-aware query refinement | Zero training | Ceiling-limited by corpus size (top excerpt didn't change), but the relevance *score* improved every time — expected to matter more at larger corpus scale |
| 2. LoRA fine-tune | ~150 iters, 40 examples, minutes | Fixed format compliance (VERDICT tag reliability), did **not** fix the underlying reasoning at this data scale — a scoped negative result, not a dead end |
| 3. Domain-adaptive embedding fine-tune | ~16s training, 100ish pairs | Clean, unambiguous improvement in chunk-level retrieval accuracy |

Because idea 2's training-based fix under-delivered, the RAG-CAV loop's actual production fix for the same underlying bug (`15_case2_cav_rag_loop.ipynb`, Part 4) is a deterministic, code-computed verdict instead — removing the LLM from the judgment entirely rather than trying to train the judgment into it. Full detail in `docs/project-summary.md` §3.7.
